# AMEX Enterprise Credit Risk Platform
## Notebook 56 -- Credit Line Management: Validation & Deployment
### Phase 4 . Problem Statement 10: Credit Line Management

CRISP-DM stage: **Evaluation & Deployment**. Depends on Problem 1 Notebooks 01-05 (real champion model),
Problem 6 Notebooks 38-40 (real persisted trailing-window model), and this problem's own Notebook 54
(real policy) and Notebook 55 (real modeling results + real persisted worklist).

**What this notebook does:** independently REPRODUCES Notebook 55's entire pipeline from scratch --
re-scores static and dynamic PD, re-composes `PD_TREND`, re-fits the tertile cuts on the real TRAIN
split, and re-validates both hard-gating KPIs on the real HOLDOUT split -- then cross-checks the fresh
reproduction against Notebook 55's persisted `credit_line_worklist.parquet` on a real sample of holdout
customers, so a silent drift between what Notebook 55 reported and what it actually persisted cannot
pass unnoticed. It then bootstraps real confidence intervals (200 resamples) on both hard-gating KPIs so
the pass/fail call accounts for real sampling uncertainty rather than a single point estimate, generates
`credit_line_scoring_service.py` (a FastAPI microservice with API-key auth and deterministic
rule-narration explainability), drives that generated service with a live self-test against 3 real
sampled holdout customers, and writes the final `credit_line_deployment_policy.json` recommendation.

**Architecture decision -- composed microservice, not a third feature pipeline:** the generated service
takes `static_pd` and `dynamic_pd` as direct request inputs rather than re-implementing Problem 1's
(~200-feature) and Problem 6's (243-feature) full feature-engineering and encoding pipelines a third
time. Reimplementing both here would duplicate ~450 combined feature fields across three services and
create a drift risk every time either upstream model is retrained -- the same anti-duplication reasoning
already applied in this platform's Notebook 46 Section 4 and Notebook 50 Section 4. Callers are assumed
to have already scored `static_pd` via Problem 1's deployed service and `dynamic_pd` via Problem 6's.

**Independent reproduction, not a re-read of Notebook 55's numbers:** every value this notebook checks
against Notebook 55's reported results is recomputed live from the real raw CSVs and the real persisted
Problem 1 / Problem 6 models, not read back from `credit_line_modeling_results.json` and assumed correct
-- matching this platform's established Notebook 52 validation pattern. `build_trailing_window_store()`
is called exactly ONCE for the single trailing-window population this notebook needs (there is no
per-split pair here, so the double-scan bug class fixed in Notebook 52 does not apply to this notebook's
own code -- noted explicitly at the call site).

**HYPER note:** Sections 1-3's structure and Section 4's reproduction pattern reuse this platform's
established Phase 4 / Notebook 52 templates verbatim.

**WARP note:** same Phase 4 tightened 92%/92% CPU/RAM cap and two-tier RAM pre-flight guard as Notebooks
54-55. Section 4's `build_trailing_window_store()` call is this notebook's single heaviest step -- RSS
and available-RAM checkpoints are printed immediately before and after it.

Zero-fabrication statement: every number this notebook prints is either computed live against the real
raw Kaggle CSVs and the real persisted models from Problems 1 and 6, a real bootstrap resample, a real
live API self-test response, or an explicitly labeled ASSUMPTION carried forward from Notebook 54 -- no
results are hardcoded or estimated in advance.


In [ ]:
# =============================================================================
# SECTION 1: ENVIRONMENT SETUP -- LOAD NOTEBOOKS 54/55'S REAL POLICY AND
#            RESULTS
# =============================================================================
import os
import sys
import gc
import json
import time
import importlib.util
import warnings
from pathlib import Path
from datetime import datetime, timezone


def _section(title: str) -> None:
    bar = "=" * 78
    print(f"\n{bar}\n{title}\n{bar}")


_section("SECTION 1: Environment Setup -- Load Notebooks 54/55's Real Policy and Results")

PROJECT_ROOT = Path(r"C:\Users\rnand\Downloads\amex-default-prediction\AMEX_Enterprise_Credit_Risk_Platform")
ARTIFACTS_DIR = PROJECT_ROOT / "artifacts"
CONFIG_PATH = ARTIFACTS_DIR / "project_config.json"
NB02_SUMMARY_PATH = ARTIFACTS_DIR / "notebook_02_summary.json"
NB04_SUMMARY_PATH = ARTIFACTS_DIR / "notebook_04_summary.json"
NB05_SUMMARY_PATH = ARTIFACTS_DIR / "notebook_05_summary.json"
NB40_SUMMARY_PATH = ARTIFACTS_DIR / "notebook_40_summary.json"
NB54_SUMMARY_PATH = ARTIFACTS_DIR / "notebook_54_summary.json"
NB55_SUMMARY_PATH = ARTIFACTS_DIR / "notebook_55_summary.json"

for _p, _fix in [
    (CONFIG_PATH, "run 01_business_understanding.ipynb first"),
    (NB02_SUMMARY_PATH, "run 02_data_engineering.ipynb first"),
    (NB04_SUMMARY_PATH, "run 04_feature_engineering.ipynb first"),
    (NB05_SUMMARY_PATH, "run 05_model_development.ipynb first"),
    (NB40_SUMMARY_PATH, "run 40_dynamic_behavioral_scoring_validation_deployment.ipynb first"),
    (NB54_SUMMARY_PATH, "run 54_credit_line_management_business_understanding.ipynb first"),
    (NB55_SUMMARY_PATH, "run 55_credit_line_management_modeling.ipynb first"),
]:
    if not _p.exists():
        raise FileNotFoundError(f"{_p} not found.\nFix: {_fix}")

with open(CONFIG_PATH, "r", encoding="utf-8") as f:
    PROJECT_CONFIG = json.load(f)
with open(NB02_SUMMARY_PATH, "r", encoding="utf-8") as f:
    NB02_SUMMARY = json.load(f)
with open(NB04_SUMMARY_PATH, "r", encoding="utf-8") as f:
    NB04_SUMMARY = json.load(f)
with open(NB05_SUMMARY_PATH, "r", encoding="utf-8") as f:
    NB05_SUMMARY = json.load(f)
with open(NB40_SUMMARY_PATH, "r", encoding="utf-8") as f:
    NB40_SUMMARY = json.load(f)
with open(NB54_SUMMARY_PATH, "r", encoding="utf-8") as f:
    NB54_SUMMARY = json.load(f)
with open(NB55_SUMMARY_PATH, "r", encoding="utf-8") as f:
    NB55_SUMMARY = json.load(f)

POLICY_PATH = Path(NB54_SUMMARY["policy_path"])
with open(POLICY_PATH, "r", encoding="utf-8") as f:
    CREDIT_LINE_POLICY = json.load(f)
MODELING_RESULTS_PATH = Path(NB55_SUMMARY["modeling_results_path"])
with open(MODELING_RESULTS_PATH, "r", encoding="utf-8") as f:
    MODELING_RESULTS = json.load(f)
WORKLIST_PATH = Path(NB55_SUMMARY["worklist_path"])
if not WORKLIST_PATH.exists():
    raise FileNotFoundError(f"{WORKLIST_PATH} not found.\nFix: re-run Notebook 55.")

RISK_LEVEL_NAMES = CREDIT_LINE_POLICY["risk_level_names"]
TREND_NAMES = CREDIT_LINE_POLICY["trend_names"]
ACTION_TIER_MATRIX = CREDIT_LINE_POLICY["kpi_targets"]["action_tier_policy"]["matrix"]
KPI_TARGETS = CREDIT_LINE_POLICY["kpi_targets"]
REPORTED_RISK_LEVEL_RATIO = MODELING_RESULTS["kpi_results"]["risk_level_monotonicity"]["top_to_bottom_ratio"]
REPORTED_RISK_LEVEL_CUT_LOW = MODELING_RESULTS["risk_level_cut_low"]
REPORTED_RISK_LEVEL_CUT_HIGH = MODELING_RESULTS["risk_level_cut_high"]
REPORTED_TREND_CUT_LOW = MODELING_RESULTS["trend_cut_low"]
REPORTED_TREND_CUT_HIGH = MODELING_RESULTS["trend_cut_high"]
REPORTED_DYNAMIC_PD_ROC_AUC = MODELING_RESULTS["dynamic_pd_roc_auc"]
REPORTED_RECOMMENDED_FOR_PRODUCTION = MODELING_RESULTS["recommended_for_production"]

TRAIN_FULL_ENGINEERED_PATH = Path(NB04_SUMMARY["output_files"]["train_full_engineered.parquet"])
PILLAR_DIRS = {k: Path(v) for k, v in PROJECT_CONFIG["pillar_dirs"].items()}
CHAMPION_NAME = NB05_SUMMARY["champion_model"]
if "model_development" not in PILLAR_DIRS:
    raise KeyError("project_config.json's pillar_dirs has no 'model_development' entry.")
_p1_models_subdir = PILLAR_DIRS["model_development"] / "models"
P1_CHAMPION_MODEL_PATH = _p1_models_subdir / (CHAMPION_NAME + ".joblib")
P1_PREPROCESSING_PATH = _p1_models_subdir / "preprocessing_artifacts.joblib"
P6_WINNING_W = NB40_SUMMARY["winning_w"]
P6_MODEL_PATH = Path(NB40_SUMMARY["model_path"])
P6_PREPROCESSING_PATH = Path(NB40_SUMMARY["preprocessing_path"])
for _p, _label in [
    (TRAIN_FULL_ENGINEERED_PATH, "Problem 1's real engineered feature matrix"),
    (P1_CHAMPION_MODEL_PATH, "Problem 1's persisted champion model"),
    (P1_PREPROCESSING_PATH, "Problem 1's preprocessing artifacts"),
    (P6_MODEL_PATH, "Problem 6's persisted model"),
    (P6_PREPROCESSING_PATH, "Problem 6's preprocessing artifacts"),
]:
    if not _p.exists():
        raise FileNotFoundError(f"{_p} not found ({_label}).")

RANDOM_SEED = PROJECT_CONFIG["random_seed"]
DETECTED_LOGICAL_CORES = PROJECT_CONFIG["hardware"]["logical_cores_detected"]
DETECTED_TOTAL_RAM_BYTES = PROJECT_CONFIG["resource_limits"]["total_ram_bytes_detected"]
EAD_PER_ACCOUNT_USD = CREDIT_LINE_POLICY["ead_per_account_usd"]
LGD_ASSUMPTION = CREDIT_LINE_POLICY["lgd_assumption"]

P10_ROOT = PROJECT_ROOT / "Phase4_Operational_Risk_Management" / "10_Problem10_Credit_Line_Management"
if "credit_line_validation_deployment" in PILLAR_DIRS:
    VALIDATION_DIR = PILLAR_DIRS["credit_line_validation_deployment"]
else:
    VALIDATION_DIR = P10_ROOT / "validation_deployment"
    print(f"NOTE: 'credit_line_validation_deployment' not in pillar_dirs -- using fallback: {VALIDATION_DIR}")
VALIDATION_DIR.mkdir(parents=True, exist_ok=True)
API_SUBDIR = P10_ROOT / "src"
API_SUBDIR.mkdir(parents=True, exist_ok=True)
DOCS_SUBDIR = P10_ROOT / "docs"
DOCS_SUBDIR.mkdir(parents=True, exist_ok=True)

print(f"Loaded policy from : {POLICY_PATH}")
print(f"Loaded results from: {MODELING_RESULTS_PATH}")
print(f"Reported DYNAMIC_PD ROC-AUC (Notebook 55): {REPORTED_DYNAMIC_PD_ROC_AUC:.4f} "
      f"(recommended_for_production: {REPORTED_RECOMMENDED_FOR_PRODUCTION})")
print(f"Validation artifacts will be written under: {VALIDATION_DIR}")
print("\n✅ Section 1 complete.")


# =============================================================================
# SECTION 2: WARP HARDWARE CONFIGURATION & LIBRARY IMPORTS (SAME TIGHTENED
#            92%/92% CAP + TWO-TIER RAM GUARD, REUSED VERBATIM)
# =============================================================================
_section("SECTION 2: WARP Hardware Configuration & Library Imports")

_PHASE4_CPU_FRACTION_CAP = 0.92
_PHASE4_RAM_FRACTION_CAP = 0.92
_historical_thread_count = PROJECT_CONFIG["resource_limits"]["warp_thread_count"]
_historical_max_ram_bytes = PROJECT_CONFIG["resource_limits"]["max_ram_bytes"]
WARP_THREAD_COUNT = min(_historical_thread_count, max(1, round(DETECTED_LOGICAL_CORES * _PHASE4_CPU_FRACTION_CAP)))
MAX_RAM_BYTES = min(_historical_max_ram_bytes, round(DETECTED_TOTAL_RAM_BYTES * _PHASE4_RAM_FRACTION_CAP))
os.environ["POLARS_MAX_THREADS"] = str(WARP_THREAD_COUNT)
warnings.filterwarnings("ignore", category=UserWarning)

import logging
logger = logging.getLogger("amex_platform")
logger.setLevel(logging.INFO)
if not logger.handlers:
    _handler = logging.StreamHandler(sys.stdout)
    _handler.setFormatter(logging.Formatter("%(asctime)s | %(levelname)-7s | %(message)s", "%H:%M:%S"))
    logger.addHandler(_handler)

missing = []
try:
    import polars as pl
except ImportError:
    missing.append("polars")
try:
    import numpy as np
except ImportError:
    missing.append("numpy")
try:
    import psutil
except ImportError:
    missing.append("psutil")
try:
    import joblib
except ImportError:
    missing.append("joblib")
try:
    from sklearn.metrics import roc_auc_score
except ImportError:
    missing.append("scikit-learn")
try:
    from fastapi.testclient import TestClient
except ImportError:
    missing.append("fastapi")
if missing:
    raise ImportError(
        "Missing required package(s): " + ", ".join(missing) + "\n"
        "Fix: run this in a terminal, then re-run this cell:\n"
        f"    pip install {' '.join(missing)}"
    )


def _rss_gb() -> float:
    return psutil.Process().memory_info().rss / 1e9


def _available_ram_gb() -> float:
    return psutil.virtual_memory().available / 1e9


_available_ram_gb_at_start = _available_ram_gb()
_comfortable_available_ram_gb = 0.50 * (MAX_RAM_BYTES / 1e9)
_min_required_available_ram_gb = 0.25 * (MAX_RAM_BYTES / 1e9)
if _available_ram_gb_at_start < _min_required_available_ram_gb:
    raise RuntimeError(
        f"Only {_available_ram_gb_at_start:.2f} GB of system RAM is available, below the "
        f"{_min_required_available_ram_gb:.2f} GB floor Section 4's reproduction needs. Close other "
        f"Jupyter kernels / applications, confirm with `psutil.virtual_memory().available / 1e9`, then "
        f"re-run this notebook from the top."
    )
if _available_ram_gb_at_start < _comfortable_available_ram_gb:
    print(f"⚠️  WARNING: only {_available_ram_gb_at_start:.2f} GB available "
          f"(comfortable margin {_comfortable_available_ram_gb:.2f} GB) -- proceeding.")
else:
    print(f"RAM pre-flight check passed: {_available_ram_gb_at_start:.2f} GB available.")

logger.info(f"Polars thread pool configured to {WARP_THREAD_COUNT}/{DETECTED_LOGICAL_CORES} threads "
            f"(Phase 4 tightened cap)")
print(f"Process RSS at Section 2 start: {_rss_gb():.2f} GB")
print("\n✅ Section 2 complete.")


# =============================================================================
# SECTION 3: RESOLVE REAL DATA PATHS
# =============================================================================
_section("SECTION 3: Resolve Real Data Paths")

_raw_candidates = []
if "raw_data_dir" in PROJECT_CONFIG:
    _raw_candidates.append(Path(PROJECT_CONFIG["raw_data_dir"]) / "train_data.csv")
if "data_root" in PROJECT_CONFIG:
    _raw_candidates.append(Path(PROJECT_CONFIG["data_root"]) / "train_data.csv")
_raw_candidates.append(PROJECT_ROOT.parent / "Raw Data From Kaggle" / "train_data.csv")

RAW_TRAIN_DATA_PATH = None
for _candidate in _raw_candidates:
    if _candidate.exists() and _candidate.stat().st_size > 1_000_000:
        RAW_TRAIN_DATA_PATH = _candidate
        break
if RAW_TRAIN_DATA_PATH is None:
    raise FileNotFoundError(
        "Could not find the raw train_data.csv. Checked:\n" + "\n".join(f"  - {c}" for c in _raw_candidates)
    )
RAW_TRAIN_LABELS_PATH = RAW_TRAIN_DATA_PATH.parent / "train_labels.csv"


def _resolve_pillar_file(filename: str, pillar_key: str, legacy_folder_name: str,
                          stored_path_str: str = None, min_size: int = 10_000) -> Path:
    _candidates = [
        PROJECT_ROOT / "Phase1_Foundation" / "01_Problem1_Credit_Scoring_PD_Prediction"
        / legacy_folder_name / filename,
    ]
    if pillar_key in PILLAR_DIRS:
        _candidates.append(PILLAR_DIRS[pillar_key] / filename)
    _candidates.append(PROJECT_ROOT / legacy_folder_name / filename)
    if stored_path_str:
        _candidates.append(Path(stored_path_str))
    for _c in _candidates:
        if _c.exists() and _c.stat().st_size > min_size:
            return _c
    raise FileNotFoundError(
        f"Could not resolve a real, non-trivial {filename}. Checked:\n"
        + "\n".join(f"  - {c}" for c in _candidates)
    )


TRAIN_SPLIT_PATH = _resolve_pillar_file(
    "train_split.csv", "data_engineering", "Data_Engineering",
    stored_path_str=NB02_SUMMARY.get("output_files", {}).get("train_split.csv"),
)
TEST_SPLIT_PATH = _resolve_pillar_file(
    "test_split.csv", "data_engineering", "Data_Engineering",
    stored_path_str=NB02_SUMMARY.get("output_files", {}).get("test_split.csv"),
)
print(f"Raw train_data.csv  : {RAW_TRAIN_DATA_PATH}")
print(f"train_split.csv     : {TRAIN_SPLIT_PATH}")
print(f"test_split.csv      : {TEST_SPLIT_PATH}")
print("\n✅ Section 3 complete.")


# =============================================================================
# SECTION 4: INDEPENDENT REPRODUCTION OF NOTEBOOK 55'S PIPELINE
# =============================================================================
_section("SECTION 4: Independent Reproduction of Notebook 55's Pipeline")

# --- Rebuilds Notebook 55's entire real pipeline from scratch, in a fresh
#     kernel: re-scores STATIC_PD and DYNAMIC_PD, re-derives PD_TREND,
#     re-fits the tertile cuts, and re-validates both hard-gating KPIs --
#     the same independent-reproduction integrity check Notebooks 48/52
#     already established for Problems 8/9. Unlike Notebook 52's first
#     version (a real double-CSV-scan bug, found and fixed 2026-08-26),
#     build_trailing_window_store() below is called exactly ONCE -- there is
#     only one trailing-window population to build here (not a per-split
#     TRAIN/HOLDOUT pair), so that specific bug class does not apply to this
#     notebook's structure. ---
P1_PREPROCESSING = joblib.load(P1_PREPROCESSING_PATH)
P1_LABEL_ENCODERS = P1_PREPROCESSING["label_encoders"]
P1_FEATURE_MEDIANS = P1_PREPROCESSING["feature_medians"]
P1_ALL_FEATURE_COLS = P1_PREPROCESSING["all_feature_cols"]
P1_CATEGORICAL_COLS = P1_PREPROCESSING["categorical_encode_cols"]
P1_NUMERIC_COLS = P1_PREPROCESSING["numeric_feature_cols"]
P1_CHAMPION_USES_SCALED = CHAMPION_NAME == "logistic_regression"
if P1_CHAMPION_USES_SCALED:
    P1_SCALER = P1_PREPROCESSING["scaler"]

print(f"Before Problem 1 scoring -- RSS {_rss_gb():.2f} GB, available RAM {_available_ram_gb():.2f} GB")
_t0 = time.time()
_train_full_eng = pl.read_parquet(TRAIN_FULL_ENGINEERED_PATH, columns=["customer_ID"] + P1_ALL_FEATURE_COLS)
_cat_exprs = []
for _c in P1_CATEGORICAL_COLS:
    _classes = P1_LABEL_ENCODERS[_c]["classes"]
    _mapping = {cat: idx for idx, cat in enumerate(_classes)}
    _default = _mapping.get("__missing__", -1)
    _cat_exprs.append(
        pl.col(_c).cast(pl.Utf8).fill_null("__missing__")
        .replace_strict(_mapping, default=_default).cast(pl.Float32).alias(_c)
    )
_num_exprs = [
    pl.when(pl.col(_c).is_infinite() | pl.col(_c).is_nan()).then(None).otherwise(pl.col(_c))
    .fill_null(P1_FEATURE_MEDIANS[_c]).cast(pl.Float32).alias(_c)
    for _c in P1_NUMERIC_COLS
]
_train_full_eng = _train_full_eng.with_columns(_cat_exprs + _num_exprs)
_X_static = _train_full_eng.select(P1_ALL_FEATURE_COLS).to_numpy().astype(np.float32, copy=False)
_static_customer_ids = _train_full_eng.select("customer_ID")
del _train_full_eng
gc.collect()

P1_CHAMPION_MODEL = joblib.load(P1_CHAMPION_MODEL_PATH)
if P1_CHAMPION_USES_SCALED:
    _X_static = (_X_static - np.asarray(P1_SCALER["mean"], dtype=np.float32)) / np.asarray(
        P1_SCALER["std"], dtype=np.float32)
_static_proba = P1_CHAMPION_MODEL.predict_proba(_X_static)[:, 1]
STATIC_PD_DF = _static_customer_ids.with_columns(pl.Series("STATIC_PD", _static_proba, dtype=pl.Float64))
del _X_static, _static_proba, _static_customer_ids
gc.collect()
print(f"Problem 1 (STATIC_PD) reproduced in {time.time() - _t0:.1f}s -- {STATIC_PD_DF.height:,} customers. "
      f"RSS {_rss_gb():.2f} GB, available RAM {_available_ram_gb():.2f} GB")

P6_PREPROCESSING = joblib.load(P6_PREPROCESSING_PATH)
P6_FEATURE_MEDIANS = P6_PREPROCESSING["feature_medians"]
P6_ALL_FEATURE_COLS = P6_PREPROCESSING["all_feature_cols"]
P6_BASE_FEATURE_COLUMNS = P6_PREPROCESSING["base_feature_columns"]


def build_trailing_window_store(csv_path: Path, base_cols: list, w: int, k: int = 0) -> "pl.DataFrame":
    """Copied VERBATIM from Notebook 39 (Problem 6) / Notebook 55 (Problem 10), per this platform's
    established convention of copying reusable feature-engineering logic rather than importing it.
    k=0 (default) is each customer's LAST w statements; k=w gives the immediately-preceding,
    non-overlapping w-statement window (2026-08-27 PD_TREND redefinition, see Notebook 54 Section 6
    addendum)."""
    schema_overrides = {"customer_ID": pl.Utf8, "S_2": pl.Utf8}
    for c in base_cols:
        schema_overrides[c] = pl.Float32
    _inf_clean_exprs = [
        pl.when(pl.col(c).is_infinite()).then(None).otherwise(pl.col(c)).alias(c)
        for c in base_cols
    ]
    lf = (
        pl.scan_csv(str(csv_path), schema_overrides=schema_overrides)
        .with_columns(pl.col("S_2").str.to_date("%Y-%m-%d"))
        .with_columns(_inf_clean_exprs)
        .sort(["customer_ID", "S_2"])
        .with_columns(
            pl.len().over("customer_ID").alias("_n_statements"),
            pl.int_range(pl.len()).over("customer_ID").alias("_row_idx"),
        )
        .filter(
            (pl.col("_row_idx") >= (pl.col("_n_statements") - w - k))
            & (pl.col("_row_idx") < (pl.col("_n_statements") - k))
        )
        .with_columns(pl.int_range(pl.len()).over("customer_ID").cast(pl.Float32).alias("_t_idx"))
    )
    agg_exprs = [pl.len().alias("_actual_window_len")]
    for c in base_cols:
        agg_exprs += [
            pl.cov(pl.col("_t_idx"), pl.col(c)).alias(f"_cov_{c}"),
            pl.when(pl.col(c).is_not_null()).then(pl.col("_t_idx")).otherwise(None).var().alias(f"_var_t_{c}"),
            pl.col(c).first().alias(f"_first_{c}"),
            pl.col(c).last().alias(f"{c}_last"),
        ]
    grouped = lf.group_by("customer_ID", maintain_order=False).agg(agg_exprs)
    _trend_exprs = []
    for c in base_cols:
        _trend_exprs.append(
            pl.when((pl.col(f"_var_t_{c}").is_not_null()) & (pl.col(f"_var_t_{c}") > 0))
            .then(pl.col(f"_cov_{c}") / pl.col(f"_var_t_{c}")).otherwise(None).alias(f"{c}_trend_slope")
        )
        _trend_exprs.append((pl.col(f"{c}_last") - pl.col(f"_first_{c}")).alias(f"{c}_trend_delta"))
    _keep_cols = ["customer_ID", "_actual_window_len"]
    _keep_cols += [f"{c}_last" for c in base_cols]
    _keep_cols += [f"{c}_trend_slope" for c in base_cols] + [f"{c}_trend_delta" for c in base_cols]
    result = grouped.with_columns(_trend_exprs).select(_keep_cols).sort("customer_ID")
    return result.collect(engine="streaming")


print(f"\nBefore Problem 6 trailing-window build (heaviest step) -- RSS {_rss_gb():.2f} GB, "
      f"available RAM {_available_ram_gb():.2f} GB")
_t0 = time.time()
_p6_store = build_trailing_window_store(RAW_TRAIN_DATA_PATH, P6_BASE_FEATURE_COLUMNS, P6_WINNING_W)
_p6_store = _p6_store.filter(pl.col("_actual_window_len") == P6_WINNING_W)
_p6_num_exprs = [
    pl.when(pl.col(_c).is_infinite() | pl.col(_c).is_nan()).then(None).otherwise(pl.col(_c))
    .fill_null(P6_FEATURE_MEDIANS[_c]).cast(pl.Float32).alias(_c)
    for _c in P6_ALL_FEATURE_COLS
]
_p6_store = _p6_store.with_columns(_p6_num_exprs)
_X_dynamic = _p6_store.select(P6_ALL_FEATURE_COLS).to_numpy().astype(np.float32, copy=False)
_dynamic_customer_ids = _p6_store.select("customer_ID")
del _p6_store
gc.collect()

P6_MODEL = joblib.load(P6_MODEL_PATH)
_dynamic_proba = P6_MODEL.predict_proba(_X_dynamic)[:, 1]
DYNAMIC_PD_DF = _dynamic_customer_ids.with_columns(pl.Series("DYNAMIC_PD", _dynamic_proba, dtype=pl.Float64))
del _X_dynamic, _dynamic_proba, _dynamic_customer_ids
gc.collect()
print(f"Problem 6 (DYNAMIC_PD) reproduced in {time.time() - _t0:.1f}s -- {DYNAMIC_PD_DF.height:,} customers. "
      f"RSS {_rss_gb():.2f} GB, available RAM {_available_ram_gb():.2f} GB")

# --- 2026-08-27: reproduce the real EARLIER, non-overlapping window too (same reused function, same
#     persisted Problem 6 model) -- PD_TREND is now DYNAMIC_PD minus DYNAMIC_PD_EARLY, replacing the
#     original DYNAMIC_PD - STATIC_PD cross-model residual (see Notebook 54 Section 6 addendum). ---
_t0 = time.time()
_p6_store_early = build_trailing_window_store(RAW_TRAIN_DATA_PATH, P6_BASE_FEATURE_COLUMNS, P6_WINNING_W, k=P6_WINNING_W)
_p6_store_early = _p6_store_early.filter(pl.col("_actual_window_len") == P6_WINNING_W)
_p6_early_num_exprs = [
    pl.when(pl.col(_c).is_infinite() | pl.col(_c).is_nan()).then(None).otherwise(pl.col(_c))
    .fill_null(P6_FEATURE_MEDIANS[_c]).cast(pl.Float32).alias(_c)
    for _c in P6_ALL_FEATURE_COLS
]
_p6_store_early = _p6_store_early.with_columns(_p6_early_num_exprs)
_X_dynamic_early = _p6_store_early.select(P6_ALL_FEATURE_COLS).to_numpy().astype(np.float32, copy=False)
_dynamic_early_customer_ids = _p6_store_early.select("customer_ID")
del _p6_store_early
gc.collect()

_dynamic_early_proba = P6_MODEL.predict_proba(_X_dynamic_early)[:, 1]
DYNAMIC_PD_EARLY_DF = _dynamic_early_customer_ids.with_columns(
    pl.Series("DYNAMIC_PD_EARLY", _dynamic_early_proba, dtype=pl.Float64)
)
del _X_dynamic_early, _dynamic_early_proba, _dynamic_early_customer_ids
gc.collect()
print(f"Problem 6 (DYNAMIC_PD_EARLY) reproduced in {time.time() - _t0:.1f}s -- "
      f"{DYNAMIC_PD_EARLY_DF.height:,} customers. RSS {_rss_gb():.2f} GB, "
      f"available RAM {_available_ram_gb():.2f} GB")

TARGET_DF = pl.read_csv(RAW_TRAIN_LABELS_PATH, schema_overrides={"customer_ID": pl.Utf8, "target": pl.Int8})
SCORED_DF = (
    STATIC_PD_DF.join(DYNAMIC_PD_DF, on="customer_ID", how="inner")
    .join(DYNAMIC_PD_EARLY_DF, on="customer_ID", how="inner")
    .join(TARGET_DF, on="customer_ID", how="inner")
    .with_columns((pl.col("DYNAMIC_PD") - pl.col("DYNAMIC_PD_EARLY")).alias("PD_TREND"))
)
TRAIN_IDS_DF = pl.read_csv(TRAIN_SPLIT_PATH, schema_overrides={"customer_ID": pl.Utf8}).select("customer_ID")
TEST_IDS_DF = pl.read_csv(TEST_SPLIT_PATH, schema_overrides={"customer_ID": pl.Utf8}).select("customer_ID")
SCORED_TRAIN_DF = SCORED_DF.join(TRAIN_IDS_DF, on="customer_ID", how="inner")
SCORED_HOLDOUT_DF = SCORED_DF.join(TEST_IDS_DF, on="customer_ID", how="inner")

_p_lo = CREDIT_LINE_POLICY["risk_level_cut_percentiles"][0] / 100.0
_p_hi = CREDIT_LINE_POLICY["risk_level_cut_percentiles"][1] / 100.0
REPRODUCED_RISK_LEVEL_CUT_LOW = float(SCORED_TRAIN_DF["DYNAMIC_PD"].quantile(_p_lo))
REPRODUCED_RISK_LEVEL_CUT_HIGH = float(SCORED_TRAIN_DF["DYNAMIC_PD"].quantile(_p_hi))
_t_lo = CREDIT_LINE_POLICY["trend_cut_percentiles"][0] / 100.0
_t_hi = CREDIT_LINE_POLICY["trend_cut_percentiles"][1] / 100.0
REPRODUCED_TREND_CUT_LOW = float(SCORED_TRAIN_DF["PD_TREND"].quantile(_t_lo))
REPRODUCED_TREND_CUT_HIGH = float(SCORED_TRAIN_DF["PD_TREND"].quantile(_t_hi))


def _assign_tiers(df: "pl.DataFrame") -> "pl.DataFrame":
    _risk_expr = (
        pl.when(pl.col("DYNAMIC_PD") <= REPRODUCED_RISK_LEVEL_CUT_LOW).then(pl.lit(RISK_LEVEL_NAMES[0]))
        .when(pl.col("DYNAMIC_PD") <= REPRODUCED_RISK_LEVEL_CUT_HIGH).then(pl.lit(RISK_LEVEL_NAMES[1]))
        .otherwise(pl.lit(RISK_LEVEL_NAMES[2])).alias("RISK_LEVEL")
    )
    _trend_expr = (
        pl.when(pl.col("PD_TREND") <= REPRODUCED_TREND_CUT_LOW).then(pl.lit(TREND_NAMES[0]))
        .when(pl.col("PD_TREND") <= REPRODUCED_TREND_CUT_HIGH).then(pl.lit(TREND_NAMES[1]))
        .otherwise(pl.lit(TREND_NAMES[2])).alias("TREND")
    )
    return df.with_columns([_risk_expr, _trend_expr])


SCORED_HOLDOUT_DF = _assign_tiers(SCORED_HOLDOUT_DF)
_holdout_by_risk = SCORED_HOLDOUT_DF.group_by("RISK_LEVEL").agg(pl.col("target").mean().alias("default_rate"))
_risk_default_rates = {r["RISK_LEVEL"]: r["default_rate"] for r in _holdout_by_risk.iter_rows(named=True)}
_rates_in_order = [_risk_default_rates[_n] for _n in RISK_LEVEL_NAMES]
REPRODUCED_RISK_LEVEL_RATIO = (_rates_in_order[-1] / _rates_in_order[0]) if _rates_in_order[0] > 0 else float("inf")

_y_holdout = SCORED_HOLDOUT_DF["target"].to_numpy()
_p_holdout = SCORED_HOLDOUT_DF["DYNAMIC_PD"].to_numpy()
REPRODUCED_DYNAMIC_PD_ROC_AUC = float(roc_auc_score(_y_holdout, _p_holdout))

print(f"\nReproduced DYNAMIC_PD ROC-AUC : {REPRODUCED_DYNAMIC_PD_ROC_AUC:.6f}")
print(f"Reported DYNAMIC_PD ROC-AUC   : {REPORTED_DYNAMIC_PD_ROC_AUC:.6f}")
_auc_diff = abs(REPRODUCED_DYNAMIC_PD_ROC_AUC - REPORTED_DYNAMIC_PD_ROC_AUC)
print(f"Reproduced risk-level cuts    : low={REPRODUCED_RISK_LEVEL_CUT_LOW:.6f}, "
      f"high={REPRODUCED_RISK_LEVEL_CUT_HIGH:.6f}")
print(f"Reported risk-level cuts      : low={REPORTED_RISK_LEVEL_CUT_LOW:.6f}, "
      f"high={REPORTED_RISK_LEVEL_CUT_HIGH:.6f}")
_cut_diff = abs(REPRODUCED_RISK_LEVEL_CUT_LOW - REPORTED_RISK_LEVEL_CUT_LOW) + \
    abs(REPRODUCED_RISK_LEVEL_CUT_HIGH - REPORTED_RISK_LEVEL_CUT_HIGH)

# Same real random_state and deterministic Polars pipeline reproduces bit-for-bit in principle; the
# same honest 1e-4 tolerance convention this platform's other reproduction checks use (Notebook 48
# Section 4, Notebook 52 Section 4) is applied here for floating-point non-associativity across
# threaded reductions.
REPRODUCTION_PASSED = bool(_auc_diff < 1e-4 and _cut_diff < 1e-4)
print(f"\nReproduction diff (ROC-AUC): {_auc_diff:.8f}, (cuts): {_cut_diff:.8f} -- "
      f"{'PASS' if REPRODUCTION_PASSED else 'FAIL'}")
if not REPRODUCTION_PASSED:
    raise AssertionError(
        f"Notebook 55 did NOT reproduce (ROC-AUC diff {_auc_diff:.8f}, cut diff {_cut_diff:.8f}) -- do "
        f"not proceed to deployment until this is resolved."
    )

# --- Cross-check the persisted worklist artifact against this fresh
#     reproduction, for a real sample of holdout customers -- catches a
#     class of bug an aggregate-metric check alone cannot (a stale or
#     corrupted worklist file on disk, even if the aggregate metrics above
#     happen to still match). ---
_persisted_worklist = pl.read_parquet(WORKLIST_PATH)
_sample_ids = SCORED_HOLDOUT_DF.sort("customer_ID").head(200).select("customer_ID")
_repro_sample = SCORED_HOLDOUT_DF.join(_sample_ids, on="customer_ID", how="inner").select(
    ["customer_ID", "DYNAMIC_PD", "RISK_LEVEL", "TREND"]
)
_persisted_sample = _persisted_worklist.join(_sample_ids, on="customer_ID", how="inner").select(
    ["customer_ID", "DYNAMIC_PD", "RISK_LEVEL", "TREND"]
)
_compare = _repro_sample.join(
    _persisted_sample, on="customer_ID", how="inner", suffix="_persisted"
).with_columns((pl.col("DYNAMIC_PD") - pl.col("DYNAMIC_PD_persisted")).abs().alias("_dpd_diff"))
_max_sample_diff = float(_compare["_dpd_diff"].max()) if _compare.height > 0 else float("inf")
_tier_mismatches = int(
    (_compare["RISK_LEVEL"] != _compare["RISK_LEVEL_persisted"]).sum()
    + (_compare["TREND"] != _compare["TREND_persisted"]).sum()
)
WORKLIST_VERIFIED = bool(
    _compare.height == _sample_ids.height and _max_sample_diff < 1e-4 and _tier_mismatches == 0
)
print(f"\nPersisted worklist cross-check ({_compare.height} sampled holdout customers): "
      f"max DYNAMIC_PD diff {_max_sample_diff:.8f}, tier mismatches {_tier_mismatches} -- "
      f"{'PASS' if WORKLIST_VERIFIED else 'FAIL'}")
if not WORKLIST_VERIFIED:
    raise AssertionError(
        "The persisted worklist does NOT match a fresh reproduction for the sampled customers -- do not "
        "deploy this artifact until this is resolved."
    )
print("\n✅ Section 4 complete.")


# =============================================================================
# SECTION 5: BOOTSTRAP CONFIDENCE INTERVALS -- BOTH HARD-GATING KPIS
# =============================================================================
_section("SECTION 5: Bootstrap Confidence Intervals -- Both Hard-Gating KPIs")

_rng = np.random.default_rng(RANDOM_SEED)
_n_boot = 200
_n_holdout = SCORED_HOLDOUT_DF.height
_holdout_risk_arr = SCORED_HOLDOUT_DF["RISK_LEVEL"].to_numpy()
_holdout_trend_arr = SCORED_HOLDOUT_DF["TREND"].to_numpy()
_holdout_target_arr = SCORED_HOLDOUT_DF["target"].to_numpy()

_boot_ratios = np.empty(_n_boot, dtype=np.float64)
_boot_worse_minus_better = {name: np.empty(_n_boot, dtype=np.float64) for name in RISK_LEVEL_NAMES}
for _i in range(_n_boot):
    _idx = _rng.integers(0, _n_holdout, size=_n_holdout)
    _risk_b, _trend_b, _y_b = _holdout_risk_arr[_idx], _holdout_trend_arr[_idx], _holdout_target_arr[_idx]
    _rates = {}
    for _name in RISK_LEVEL_NAMES:
        _mask = _risk_b == _name
        _rates[_name] = _y_b[_mask].mean() if _mask.sum() > 0 else np.nan
    _boot_ratios[_i] = (_rates[RISK_LEVEL_NAMES[-1]] / _rates[RISK_LEVEL_NAMES[0]]
                         if _rates[RISK_LEVEL_NAMES[0]] > 0 else np.nan)
    for _name in RISK_LEVEL_NAMES:
        _mask_better = (_risk_b == _name) & (_trend_b == TREND_NAMES[0])
        _mask_worse = (_risk_b == _name) & (_trend_b == TREND_NAMES[2])
        _better_rate = _y_b[_mask_better].mean() if _mask_better.sum() > 0 else np.nan
        _worse_rate = _y_b[_mask_worse].mean() if _mask_worse.sum() > 0 else np.nan
        _boot_worse_minus_better[_name][_i] = _worse_rate - _better_rate

_boot_ratios_valid = _boot_ratios[~np.isnan(_boot_ratios)]
RISK_LEVEL_RATIO_CI = [
    float(np.percentile(_boot_ratios_valid, 2.5)), float(np.percentile(_boot_ratios_valid, 97.5))
] if len(_boot_ratios_valid) > 0 else [float("nan"), float("nan")]
print(f"Bootstrap 95% CI on risk-level top/bottom default-rate ratio "
      f"({len(_boot_ratios_valid)} valid resamples of {_n_boot}): "
      f"[{RISK_LEVEL_RATIO_CI[0]:.2f}, {RISK_LEVEL_RATIO_CI[1]:.2f}]")

TREND_COHERENCE_GAP_CI = {}
for _name in RISK_LEVEL_NAMES:
    _valid = _boot_worse_minus_better[_name][~np.isnan(_boot_worse_minus_better[_name])]
    TREND_COHERENCE_GAP_CI[_name] = [
        float(np.percentile(_valid, 2.5)), float(np.percentile(_valid, 97.5))
    ] if len(_valid) > 0 else [float("nan"), float("nan")]
    print(f"Bootstrap 95% CI on trend-coherence gap (Worse - Better default rate), {_name} "
          f"({len(_valid)} valid resamples): [{TREND_COHERENCE_GAP_CI[_name][0]:.4f}, "
          f"{TREND_COHERENCE_GAP_CI[_name][1]:.4f}]")

RISK_LEVEL_MONOTONICITY_CI_PASSED = bool(RISK_LEVEL_RATIO_CI[0] >= 1.0)
TREND_COHERENCE_CI_PASSED = bool(all(TREND_COHERENCE_GAP_CI[n][0] > 0 for n in RISK_LEVEL_NAMES))
print(f"\nrisk_level_monotonicity CI lower bound >= 1.0 (ratio CI stays above parity): "
      f"{RISK_LEVEL_MONOTONICITY_CI_PASSED}")
print(f"trend_coherence CI lower bound > 0 in every risk-level tier (gap CI stays positive): "
      f"{TREND_COHERENCE_CI_PASSED}")
print("\n✅ Section 5 complete.")


# =============================================================================
# SECTION 6: HONEST LIMITATION -- DEPLOYMENT SCOPE & ASSUMPTIONS
# =============================================================================
_section("SECTION 6: Honest Limitation -- Deployment Scope & Assumptions")

MEETS_KPI_WITH_CI = bool(RISK_LEVEL_MONOTONICITY_CI_PASSED and TREND_COHERENCE_CI_PASSED)
print(
    "DEPLOYMENT SCOPE (honest): this service composes two already-validated real PD scores (Problem 1's "
    "static, Problem 6's dynamic) into a risk-level x trend classification and an action recommendation -- "
    "it does NOT itself execute any limit change; the action tier is a business-rule recommendation for a "
    "human or an automated downstream system to act on, not a claim that any specific action has been "
    "proven to change a real financial outcome (no real limit-change/outcome data exists in this dataset -- "
    "see Notebook 54 Section 6).\n\n"
    "'Utilization-trend' in this service is Problem 10's own PD_TREND reinterpretation (dynamic PD minus "
    "the SAME model's own real, immediately-preceding, non-overlapping dynamic PD -- redefined "
    "2026-08-27, see Notebook 54 Section 6 addendum), NOT a true balance-to-credit-limit ratio -- this "
    "dataset has no such field. Every response from the generated service states this plainly rather "
    "than implying a real utilization measurement.\n\n"
    f"RECOMMENDED FOR PRODUCTION: {MEETS_KPI_WITH_CI} (both hard-gating KPIs' 95% bootstrap CIs "
    f"{'hold' if MEETS_KPI_WITH_CI else 'do NOT hold'} on the real HOLDOUT split)."
)
print("\n✅ Section 6 complete.")


# =============================================================================
# SECTION 7: PERSIST DEPLOYMENT POLICY ARTIFACT
# =============================================================================
_section("SECTION 7: Persist Deployment Policy Artifact")

CREDIT_LINE_DEPLOYMENT_POLICY = {
    "generated_at_utc": datetime.now(timezone.utc).isoformat(),
    "risk_level_names": RISK_LEVEL_NAMES,
    "trend_names": TREND_NAMES,
    "risk_level_cut_low": REPRODUCED_RISK_LEVEL_CUT_LOW,
    "risk_level_cut_high": REPRODUCED_RISK_LEVEL_CUT_HIGH,
    "trend_cut_low": REPRODUCED_TREND_CUT_LOW,
    "trend_cut_high": REPRODUCED_TREND_CUT_HIGH,
    "action_tier_matrix": ACTION_TIER_MATRIX,
    "reported_dynamic_pd_roc_auc": REPORTED_DYNAMIC_PD_ROC_AUC,
    "reproduced_dynamic_pd_roc_auc": REPRODUCED_DYNAMIC_PD_ROC_AUC,
    "reproduction_passed": REPRODUCTION_PASSED,
    "worklist_verified": WORKLIST_VERIFIED,
    "risk_level_ratio": REPRODUCED_RISK_LEVEL_RATIO,
    "risk_level_ratio_ci_95": RISK_LEVEL_RATIO_CI,
    "trend_coherence_gap_ci_95": TREND_COHERENCE_GAP_CI,
    "risk_level_monotonicity_ci_passed": RISK_LEVEL_MONOTONICITY_CI_PASSED,
    "trend_coherence_ci_passed": TREND_COHERENCE_CI_PASSED,
    "meets_kpi_with_ci": MEETS_KPI_WITH_CI,
    "recommended_for_production": bool(MEETS_KPI_WITH_CI and REPRODUCTION_PASSED and WORKLIST_VERIFIED),
    "utilization_trend_reinterpretation": CREDIT_LINE_POLICY["utilization_trend_reinterpretation"],
    "ead_per_account_usd": EAD_PER_ACCOUNT_USD,
    "lgd_assumption": LGD_ASSUMPTION,
    "random_seed": RANDOM_SEED,
}
deployment_policy_path = DOCS_SUBDIR / "credit_line_deployment_policy.json"
with open(deployment_policy_path, "w", encoding="utf-8") as f:
    json.dump(CREDIT_LINE_DEPLOYMENT_POLICY, f, indent=2)
print(f"Wrote: {deployment_policy_path}")
print("\n✅ Section 7 complete.")


# =============================================================================
# SECTION 8: GENERATE credit_line_scoring_service.py -- REAL, RUNNABLE
#            FASTAPI SERVICE WITH AUTH + EXPLAINABILITY FROM DAY ONE
# =============================================================================
_section("SECTION 8: Generate credit_line_scoring_service.py (Auth + Explainability From Day One)")

# --- Architecture note: this service takes DYNAMIC_PD (current) and DYNAMIC_PD_EARLY (the same
#     Problem 6 model, an immediately-preceding non-overlapping window) as INPUTS (already-computed,
#     e.g. from two calls to Problem 6's own real deployed /predict endpoint) rather than
#     re-implementing that model's full feature-engineering + encoding pipeline a third time.
#     Reimplementing ~200+ raw feature fields inside a THIRD service would duplicate an
#     already-deployed service's entire request contract, risking silent drift between two copies of
#     the same logic -- the exact anti-pattern this platform has repeatedly flagged (Notebook 46
#     Section 4, Notebook 50 Section 4). Composing an upstream service's own two real outputs is
#     standard microservice layering for exactly this kind of decision-composition service, and keeps
#     this one, real source of truth (the cuts + action matrix) the only thing this service actually
#     owns. (2026-08-27: this replaces the original STATIC_PD + DYNAMIC_PD composition -- see
#     Notebook 54 Section 6 addendum for why.) ---
#
#     Explainability: deterministic rule narration (which cut thresholds
#     the two real PD scores fell into, and which policy cell produced the
#     action) -- the exact-for-this-technique choice per this platform's
#     established per-model-type table (Notebook 52's own note): this is a
#     rule-based bucket-and-lookup system, not a trained classifier, so
#     occlusion-based marginal contribution does not apply here -- the same
#     reasoning already used for Problem 3's IFRS9 rule engine and Problem
#     4/8's linear-term decomposition. ---
_policy_path_str = str(deployment_policy_path)

CREDIT_LINE_SERVICE_TEMPLATE = "\n".join([
    "# AMEX Enterprise Credit Risk Platform -- Credit Line Management Recommendation API.",
    "# Auto-generated by 56_credit_line_management_validation_deployment.ipynb.",
    "# Composes a customer's real static PD (Problem 1) and real dynamic PD (Problem 6) into a risk-level",
    "# x trend classification and a real credit-line action recommendation.",
    "# Every endpoint except /health requires a valid X-API-Key header (see .env.example).",
    "# Run with:",
    "#     uvicorn credit_line_scoring_service:app --host 0.0.0.0 --port 8010",
    "import json",
    "import logging",
    "import os",
    "import secrets",
    "from pathlib import Path",
    "from typing import List, Optional",
    "",
    "from fastapi import Depends, FastAPI, HTTPException, Security",
    "from fastapi.security import APIKeyHeader",
    "from pydantic import BaseModel",
    "",
    "_auth_logger = logging.getLogger(__name__ + \".auth\")",
    "_DEV_DEFAULT_API_KEY = \"dev-only-CHANGE-ME-before-deploying\"",
    "_api_key_header = APIKeyHeader(name=\"X-API-Key\", auto_error=False)",
    "",
    "",
    "def _configured_api_key() -> str:",
    "    key = os.environ.get(\"API_KEY\")",
    "    if not key:",
    "        _auth_logger.warning(",
    "            \"API_KEY is not set -- falling back to the published dev-only default. Set API_KEY \"",
    "            \"before deploying this service anywhere reachable by anyone but you.\"",
    "        )",
    "        return _DEV_DEFAULT_API_KEY",
    "    return key",
    "",
    "",
    "def require_api_key(presented: str = Security(_api_key_header)) -> str:",
    "    expected = _configured_api_key()",
    "    if not presented or not secrets.compare_digest(presented, expected):",
    "        raise HTTPException(status_code=401, detail=\"Missing or invalid X-API-Key header.\")",
    "    return presented",
    "",
    "",
    "POLICY_PATH = Path(os.environ.get(\"AMEX_P10_POLICY_PATH\", r\"__POLICY_PATH_TOKEN__\"))",
    "with open(POLICY_PATH, \"r\", encoding=\"utf-8\") as _f:",
    "    _POLICY = json.load(_f)",
    "",
    "RISK_LEVEL_NAMES = _POLICY[\"risk_level_names\"]",
    "TREND_NAMES = _POLICY[\"trend_names\"]",
    "RISK_LEVEL_CUT_LOW = _POLICY[\"risk_level_cut_low\"]",
    "RISK_LEVEL_CUT_HIGH = _POLICY[\"risk_level_cut_high\"]",
    "TREND_CUT_LOW = _POLICY[\"trend_cut_low\"]",
    "TREND_CUT_HIGH = _POLICY[\"trend_cut_high\"]",
    "ACTION_MAP = {(c[\"risk_level\"], c[\"trend\"]): c for c in _POLICY[\"action_tier_matrix\"]}",
    "RECOMMENDED_FOR_PRODUCTION = _POLICY[\"recommended_for_production\"]",
    "UTILIZATION_TREND_NOTE = _POLICY[\"utilization_trend_reinterpretation\"]",
    "",
    "",
    "class RecommendRequest(BaseModel):",
    "    customer_id: Optional[str] = None",
    "    dynamic_pd: float",
    "    dynamic_pd_early: float",
    "",
    "",
    "class RecommendResponse(BaseModel):",
    "    customer_id: Optional[str] = None",
    "    dynamic_pd: float",
    "    dynamic_pd_early: float",
    "    pd_trend: float",
    "    risk_level: str",
    "    trend: str",
    "    action: str",
    "    rationale: str",
    "    reasoning: List[str] = []",
    "    recommended_for_production: bool = RECOMMENDED_FOR_PRODUCTION",
    "",
    "",
    "def _assign_risk_level(dynamic_pd: float) -> str:",
    "    if dynamic_pd <= RISK_LEVEL_CUT_LOW:",
    "        return RISK_LEVEL_NAMES[0]",
    "    if dynamic_pd <= RISK_LEVEL_CUT_HIGH:",
    "        return RISK_LEVEL_NAMES[1]",
    "    return RISK_LEVEL_NAMES[2]",
    "",
    "",
    "def _assign_trend(pd_trend: float) -> str:",
    "    if pd_trend <= TREND_CUT_LOW:",
    "        return TREND_NAMES[0]",
    "    if pd_trend <= TREND_CUT_HIGH:",
    "        return TREND_NAMES[1]",
    "    return TREND_NAMES[2]",
    "",
    "",
    "app = FastAPI(",
    "    title=\"AMEX Enterprise Credit Risk Platform -- Credit Line Management Recommendation API\",",
    "    description=\"Composes real static + dynamic PD scores into a risk-level x trend classification \"",
    "                \"and a real credit-line action recommendation. Every endpoint except /health requires \"",
    "                \"a valid X-API-Key header.\",",
    "    version=\"1.0.0\",",
    ")",
    "",
    "",
    "@app.get(\"/health\")",
    "def health():",
    "    return {\"status\": \"ok\"}",
    "",
    "",
    "@app.get(\"/policy-info\", dependencies=[Depends(require_api_key)])",
    "def policy_info():",
    "    return {",
    "        \"risk_level_names\": RISK_LEVEL_NAMES,",
    "        \"trend_names\": TREND_NAMES,",
    "        \"risk_level_cuts\": [RISK_LEVEL_CUT_LOW, RISK_LEVEL_CUT_HIGH],",
    "        \"trend_cuts\": [TREND_CUT_LOW, TREND_CUT_HIGH],",
    "        \"action_tier_matrix\": _POLICY[\"action_tier_matrix\"],",
    "        \"recommended_for_production\": RECOMMENDED_FOR_PRODUCTION,",
    "        \"utilization_trend_note\": UTILIZATION_TREND_NOTE,",
    "    }",
    "",
    "",
    "@app.post(\"/recommend\", response_model=RecommendResponse, dependencies=[Depends(require_api_key)])",
    "def recommend(request: RecommendRequest):",
    "    if not (0.0 <= request.dynamic_pd <= 1.0) or not (0.0 <= request.dynamic_pd_early <= 1.0):",
    "        raise HTTPException(status_code=422, detail=\"dynamic_pd and dynamic_pd_early must both be in [0, 1].\")",
    "    pd_trend = request.dynamic_pd - request.dynamic_pd_early",
    "    risk_level = _assign_risk_level(request.dynamic_pd)",
    "    trend = _assign_trend(pd_trend)",
    "    cell = ACTION_MAP.get((risk_level, trend))",
    "    if cell is None:",
    "        raise HTTPException(status_code=500, detail=f\"No action defined for ({risk_level}, {trend}).\")",
    "    reasoning = [",
    "        f\"dynamic_pd={request.dynamic_pd:.4f} -> {risk_level} \"",
    "        f\"(cuts: <= {RISK_LEVEL_CUT_LOW:.4f} Low, <= {RISK_LEVEL_CUT_HIGH:.4f} Medium, else High)\",",
    "        f\"pd_trend={pd_trend:.4f} (dynamic_pd - dynamic_pd_early, same-model two-window trend) -> {trend} \"",
    "        f\"(cuts: <= {TREND_CUT_LOW:.4f} Better, <= {TREND_CUT_HIGH:.4f} Stable, else Worse)\",",
    "        f\"({risk_level}, {trend}) -> {cell['action']}\",",
    "    ]",
    "    return RecommendResponse(",
    "        customer_id=request.customer_id, dynamic_pd=request.dynamic_pd, dynamic_pd_early=request.dynamic_pd_early,",
    "        pd_trend=pd_trend, risk_level=risk_level, trend=trend, action=cell[\"action\"],",
    "        rationale=cell[\"rationale\"], reasoning=reasoning,",
    "    )",
    "",
])
CREDIT_LINE_SERVICE_SOURCE = CREDIT_LINE_SERVICE_TEMPLATE.replace("__POLICY_PATH_TOKEN__", _policy_path_str)

service_py_path = API_SUBDIR / "credit_line_scoring_service.py"
with open(service_py_path, "w", encoding="utf-8") as f:
    f.write(CREDIT_LINE_SERVICE_SOURCE)
compile(CREDIT_LINE_SERVICE_SOURCE, str(service_py_path), "exec")
print(f"Generated {len(CREDIT_LINE_SERVICE_SOURCE.splitlines())} lines, syntax-checked OK.")
print(f"Saved -> {service_py_path}")
print("\n✅ Section 8 complete.")


# =============================================================================
# SECTION 9: LIVE SELF-TEST -- IMPORT THE GENERATED SERVICE & DRIVE IT WITH
#            REAL HOLDOUT CUSTOMERS' ACTUAL SCORES
# =============================================================================
_section("SECTION 9: Live Self-Test -- Import the Generated Service & Drive It")

os.environ["AMEX_P10_POLICY_PATH"] = str(deployment_policy_path)
_TEST_API_KEY = "pytest-only-test-key"
os.environ["API_KEY"] = _TEST_API_KEY
_spec = importlib.util.spec_from_file_location("amex_credit_line_service", str(service_py_path))
_service_module = importlib.util.module_from_spec(_spec)
_spec.loader.exec_module(_service_module)
client = TestClient(_service_module.app)
_auth_headers = {"X-API-Key": _TEST_API_KEY}

_health_resp = client.get("/health")
assert _health_resp.status_code == 200
print(f"GET /health              (no key)   -> {_health_resp.status_code}  {_health_resp.json()}")

_unauth_resp = client.get("/policy-info")
assert _unauth_resp.status_code == 401
print(f"GET /policy-info         (no key, should reject) -> {_unauth_resp.status_code}")

_info_resp = client.get("/policy-info", headers=_auth_headers)
assert _info_resp.status_code == 200
print(f"GET /policy-info         (with key) -> {_info_resp.status_code}  "
      f"recommended_for_production={_info_resp.json()['recommended_for_production']}")

# Real end-to-end check against 3 real holdout customers spanning different real cells.
_sample_rows = SCORED_HOLDOUT_DF.sort("DYNAMIC_PD").select(
    ["customer_ID", "DYNAMIC_PD", "DYNAMIC_PD_EARLY", "RISK_LEVEL", "TREND"]
)
_sample_indices = [0, _sample_rows.height // 2, _sample_rows.height - 1]
API_SELF_TEST_ROWS_PASSED = []
for _idx in _sample_indices:
    _row = _sample_rows.row(_idx, named=True)
    _resp = client.post(
        "/recommend", headers=_auth_headers,
        json={"customer_id": _row["customer_ID"], "dynamic_pd": _row["DYNAMIC_PD"],
              "dynamic_pd_early": _row["DYNAMIC_PD_EARLY"]},
    )
    assert _resp.status_code == 200, f"/recommend returned {_resp.status_code}: {_resp.text}"
    _result = _resp.json()
    _row_passed = (_result["risk_level"] == _row["RISK_LEVEL"] and _result["trend"] == _row["TREND"])
    API_SELF_TEST_ROWS_PASSED.append(_row_passed)
    print(f"POST /recommend customer={_row['customer_ID']}: API risk_level={_result['risk_level']} "
          f"(expected {_row['RISK_LEVEL']}), trend={_result['trend']} (expected {_row['TREND']}), "
          f"action={_result['action']} -- {'PASS' if _row_passed else 'FAIL'}")

_unauth_rec_resp = client.post("/recommend", json={"dynamic_pd": 0.1, "dynamic_pd_early": 0.1})
assert _unauth_rec_resp.status_code == 401, "/recommend without a key should be rejected"
print(f"POST /recommend           (no key, should reject) -> {_unauth_rec_resp.status_code}")

_bad_range_resp = client.post("/recommend", headers=_auth_headers, json={"dynamic_pd": 1.5, "dynamic_pd_early": 0.1})
assert _bad_range_resp.status_code == 422, "/recommend with an out-of-range PD should be rejected"
print(f"POST /recommend    (out-of-range PD, should reject) -> {_bad_range_resp.status_code}")

API_SELF_TEST_PASSED = bool(all(API_SELF_TEST_ROWS_PASSED))
if not API_SELF_TEST_PASSED:
    raise RuntimeError("Notebook 56's API self-test FAILED -- see checks above. Not safe to proceed.")
print("\n✅ Section 9 complete -- auth rejects unkeyed calls, all 3 real sample customers' tier "
      "assignments match direct computation, input validation rejects out-of-range scores.")


# =============================================================================
# SECTION 10: GENERATE .env.example & requirements-api.txt
# =============================================================================
_section("SECTION 10: Generate .env.example & requirements-api.txt")

_env_example = "\n".join([
    "# Copy to .env and fill in real values before deploying.",
    "API_KEY=dev-only-CHANGE-ME-before-deploying",
    f"AMEX_P10_POLICY_PATH={deployment_policy_path}",
    "",
])
(API_SUBDIR / ".env.example").write_text(_env_example, encoding="utf-8")
_requirements_api = "\n".join(["fastapi>=0.110", "uvicorn>=0.29", "pydantic>=2.0", ""])
(API_SUBDIR / "requirements-api.txt").write_text(_requirements_api, encoding="utf-8")
print(f"Wrote: {API_SUBDIR / '.env.example'}")
print(f"Wrote: {API_SUBDIR / 'requirements-api.txt'}")
print("\n✅ Section 10 complete.")


# =============================================================================
# SECTION 11: VERIFICATION -- INTEGRITY CHECKS
# =============================================================================
_section("SECTION 11: Verification -- Integrity Checks")


def _check(label, condition, detail=""):
    status = "PASS" if condition else "FAIL"
    print(f"  [{status}] {label}" + (f" -- {detail}" if detail and not condition else ""))
    return condition


_all_checks_passed = True
_all_checks_passed &= _check("Deployment policy file was written", deployment_policy_path.exists())
_all_checks_passed &= _check("Service file was written and syntax-checked", service_py_path.exists())
_all_checks_passed &= _check("Reproduction matches Notebook 55's reported values (ROC-AUC + cuts)",
                              REPRODUCTION_PASSED)
_all_checks_passed &= _check("Persisted worklist verified against a fresh reproduction sample",
                              WORKLIST_VERIFIED)
_all_checks_passed &= _check("Bootstrap risk-level ratio CI is well-formed (lower <= point <= upper)",
                              RISK_LEVEL_RATIO_CI[0] <= REPRODUCED_RISK_LEVEL_RATIO <= RISK_LEVEL_RATIO_CI[1])
_all_checks_passed &= _check("API self-test passed on all sampled real holdout customers",
                              API_SELF_TEST_PASSED)
_all_checks_passed &= _check("Reproduced risk-level cuts are correctly ordered (low < high)",
                              REPRODUCED_RISK_LEVEL_CUT_LOW < REPRODUCED_RISK_LEVEL_CUT_HIGH)
_all_checks_passed &= _check("Reproduced trend cuts are correctly ordered (low < high)",
                              REPRODUCED_TREND_CUT_LOW < REPRODUCED_TREND_CUT_HIGH)
_expected_files = [deployment_policy_path, service_py_path,
                   API_SUBDIR / ".env.example", API_SUBDIR / "requirements-api.txt"]
for _fp in _expected_files:
    _all_checks_passed &= _check(f"{_fp.name} exists and is non-empty", _fp.exists() and _fp.stat().st_size > 0)

if not _all_checks_passed:
    raise AssertionError("One or more verification checks failed -- see FAIL lines above.")
print("\n✅ Section 11 complete -- all checks passed.")


# =============================================================================
# SECTION 12: WRITE NOTEBOOK 56 SUMMARY ARTIFACT & COMPLETION
# =============================================================================
_section("SECTION 12: Write Notebook 56 Summary Artifact")

NB56_SUMMARY = {
    "notebook": "56_credit_line_management_validation_deployment.ipynb",
    "generated_at_utc": datetime.now(timezone.utc).isoformat(),
    "deployment_policy_path": str(deployment_policy_path),
    "service_py_path": str(service_py_path),
    "reproduction_passed": REPRODUCTION_PASSED,
    "worklist_verified": WORKLIST_VERIFIED,
    "reproduced_dynamic_pd_roc_auc": REPRODUCED_DYNAMIC_PD_ROC_AUC,
    "risk_level_ratio_ci_95": RISK_LEVEL_RATIO_CI,
    "trend_coherence_gap_ci_95": TREND_COHERENCE_GAP_CI,
    "meets_kpi_with_ci": MEETS_KPI_WITH_CI,
    "recommended_for_production": CREDIT_LINE_DEPLOYMENT_POLICY["recommended_for_production"],
    "api_self_test_passed": API_SELF_TEST_PASSED,
    "random_seed": RANDOM_SEED,
}
NB56_SUMMARY_PATH = ARTIFACTS_DIR / "notebook_56_summary.json"
with open(NB56_SUMMARY_PATH, "w", encoding="utf-8") as f:
    json.dump(NB56_SUMMARY, f, indent=2)
print(f"Wrote: {NB56_SUMMARY_PATH}")

_section("NOTEBOOK 56 COMPLETE")
print(f"Reproduction passed                  : {REPRODUCTION_PASSED}")
print(f"Worklist verified                    : {WORKLIST_VERIFIED}")
print(f"Reproduced DYNAMIC_PD ROC-AUC         : {REPRODUCED_DYNAMIC_PD_ROC_AUC:.4f}")
print(f"risk_level_monotonicity (CI-adjusted) : {'PASS' if RISK_LEVEL_MONOTONICITY_CI_PASSED else 'FAIL'}")
print(f"trend_coherence (CI-adjusted)         : {'PASS' if TREND_COHERENCE_CI_PASSED else 'FAIL'}")
print(f"RECOMMENDED_FOR_PRODUCTION            : {CREDIT_LINE_DEPLOYMENT_POLICY['recommended_for_production']}")
print(f"API self-test                         : {'PASSED' if API_SELF_TEST_PASSED else 'FAILED'}")
print(f"Service written to: {service_py_path}")
print(
    "\nNext: 57_credit_line_management_financial_impact_reporting_packaging.ipynb -- financial-impact "
    "reporting and final packaging for Problem 10, closing out Phase 4's second problem."
)
